# Teaching Computers to Read: An Introduction to Natural Language Processing

<div class="alert alert-info">

Welcome! This activity is part of an introduction to computational notebooks, designed specifically for K-12 educators.

Here, we'll dive into a specific, powerful type of analysis: **teaching computers to work with written language**. This is called **Natural Language Processing**, or **NLP**.

</div>

Text data is everywhere: emails, social media posts, student writing, news articles, and now, text generated by AI itself. The same tools we'll use here also power things much closer to a classroom: comparing the stylistic differences between famous authors, analyzing the language of a novel, or spotting the persuasive techniques in a piece of marketing.


<hr style="border: 5px solid #003262;" />


## Key Ideas in Natural Language Processing

To explore some of the key ideas related to the code, math, and science of NLP, in this notebook we will be exploring a real dataset of **human- and AI-written text**.

**Our Learning Goals:**

* 🧹 **Cleaning and Tokenizing Text:** Learn to preprocess raw text into clean, structured `tokens` a computer can count and compare, using both hand-written code and Python's regular expressions.

* 🔢 **Counting and Weighting Words:** Use `bag of words` to count frequencies and Term Frequency-Inverse Document Frequency (`TF-IDF`) to turn text into numbers, and learn how to categorize different kinds of text using those numbers. Determine what those numbers capture well, and what they miss.

* 🤖 **AI vs. Human Content:** Explore how NLP is used to detect AI-generated writing, uncover patterns linked to real writing behavior, and consider the fairness and limits of these tools when applied to real students.


<hr style="border: 1px solid #fdb515;" />

We'll start by loading the real dataset we'll use throughout this notebook: real news articles, alongside AI-generated rewrites of those same stories from six different advanced language models (Mistral-7B, Llama-8B, Qwen-2-72B, GPT-4o, Yi-Large, and Gemma-2-9B).

**Note:** Keep the file `ai_v_human_nlp_dataset.csv` in the same folder as this notebook so the cell below can find it.


---

### Part I: Loading the Data
Remember to run each code cell, to load the data and set the variables you need to do your analysis.


**Step 1: Read the file.**

Questions for you while you run the cell below:
*   Why must we drop rows with missing text before feeding them into an NLP model?

*   How might deleting these rows introduce bias or distort our dataset?





In [ ]:
import pandas as pd # used for working with data tables

texts = pd.read_csv("ai_v_human_nlp_dataset.csv") # read and load data
texts = texts.dropna(subset=["Text"]) # drop rows with missing text

texts.shape # outputs number of rows, number of columns


**Step 2: Take a smaller sample.**

From the above output, we learned that our full file has over 17000 rows, or in other words 17000+ news articles are in our data. To be efficient as possibe, we will take 3000 random rows/articles.

Note: Notice how below of all the data, there is a subtext shows how many rows and columns we have as well.

In [ ]:
texts = texts.sample(3000, random_state=42).reset_index(drop=True)
texts # displays the first 5 records and the last 5 records

**Step 3: Clean up the columns.**

Looking at the raw data above, the column names aren't very descriptive, and the target labels use numeric codes (0 and 1). Let's rename the columns so they are easier to work with, and map the 0/1 values to readable categories (Human/AI). We will also remove the "Unnamed: 0	" column as the article ID does not contribute anything to our training.


In [ ]:
texts = texts.rename(columns={"Text": "text", "Label_B": "source_model"}) # replace the column names
texts["label"] = texts["Label_A"].replace({0: "Human", 1: "AI"}) # replace labels with readable categories
texts = texts[["text", "label", "source_model"]]

texts # this will show the first 5 records and the last 5 records


<div class="alert alert-success">

**Considering the data above…**

* What do you notice and wonder about this dataset?

* What questions could you explore further?

* What extra information would help you interpret these labels?

</div>


Here are the measurements included with each text, along with what they represent:

| Column | Description |
| :--- | :--- |
| **`text`** | The full article: either the original human-written story, or an AI-generated rewrite of it. |
| **`label`** | Whether a human or an AI wrote the text. |
| **`source_model`** | Which of six language models wrote the text, or `Human_Story` if it's the real, human-written article. |

Our focus in Parts II and III is primarly turning messy, everyday text into something a computer can measure, which is the same first step behind spell-checkers, spam filters, and search engines.

**Note:** The examples we will be using will be straight from this dataset.



---

### Part II: Getting Text Ready

In programming, text is stored as a string: a chain of characters made up of letters, numbers, symbols, and spaces. These real text can get messy fast, with its different capitalization, punctuation, extra spaces.

Before a computer can count or compare words within a string, we usually clean it up first, a step we called **preprocessing**.

Let's grab one example from our dataset to work with (a **Human**-written sentence that happens to include a contraction).


**Step 1: Find an example sentence.**

We'll search through the human-written texts for one clean, medium-length sentence that contains a contraction (like "doesn't" or "party's").

**Note:** To learn more about the **regular expression `re`** library, check out this [link](https://docs.python.org/3/library/re.html).

In [ ]:
import re # re is Python's built-in module for pattern matching in text

def find_example_sentence():
    for text in texts.loc[texts["label"] == "Human", "text"]:
        for sentence in re.split(r"(?<=[.!?])\s+", text):
            if 80 <= len(sentence) <= 220 and re.search(r"[a-z][’'](?:t|ll|re|ve|s|d)\b", sentence.lower()):
                return sentence.strip()

my_sentence = find_example_sentence()
my_sentence

**Step 2: Converting all text to lowercase.**

Computers treat "The" and "the" as completely different words. Converting everything to lowercase prevents our model from treating the same word as distinct entries just because of capitalization.


In [ ]:
lowercased = my_sentence.lower()
lowercased


**Step 3: Remove all punctuations.**

The simplest way to isolate words is to just remove anything that isn't a letter or a space. These non letters and space characters are includes periods, commas, quotation marks, and other symbols.


In [ ]:
punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~’‘“”' # all symbols on the keyboard

cleaned = "".join(char for char in lowercased if char not in punctuation)
cleaned


**Step 4: Split into individual words.**

With punctuation removed, we can split the text on whitespace to create `tokens`, which are individual words our model can now count and analyze.


In [ ]:
naive_tokens = cleaned.split()
naive_tokens

<div class="alert alert-success">

**Considering the result above…**

* Find the contraction in your sentence (in the original, before cleaning). What happened to it after we removed punctuation? Is that a problem?

* Looking at the list of naive tokens: which words seem to carry the most information about what this sentence is about? Which carry the least?

* What questions do you still have about what we've done here?

</div>


You may have noticed that removing punctuation turned "i'll" into "ill" — we lost the apostrophe and created a new, misspelled word. A smarter approach handles this more carefully.

Let's compare our naive approach to tokenizing with **regular expressions** (`re`), a built-in Python tool for matching patterns in text.


In [ ]:
def regex_tokenize(text):
    # This pattern matches runs of letters and internal apostrophes (both the
    # straight kind (') and the curly kind (’) that human writing often uses)
    # so a contraction like "i'll" stays whole instead of becoming "ill".
    return re.findall(r"[a-z’']+", text.lower())

regex_tokens = regex_tokenize(my_sentence)

print("Naive split:   ", naive_tokens)
print("Regex tokenize:", regex_tokens)

In most English text, "this" and "it" are called **stop words**, these words often builds noise to the algorithm then providing more understanding to. Conveniently, the library `scikit-learn` comes with a ready-made stop word list, so we can use it to remove.

**Note:** We will exlore more of `scikit-learn` later in the notebook, but to learn about this particular section, see this [link](https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/feature_extraction/_stop_words.py).

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

content_words = [w for w in regex_tokens if w not in ENGLISH_STOP_WORDS]
content_words


<div class="alert alert-success">

**Comparing the three versions of this sentence…**

* Naive split, regex tokens, and regex tokens with stop words removed each look a little different. For a task like counting how often "good" appears, does it matter which one you use?

* For a task involving negation or contractions, why might the naive version cause more trouble?

</div>

**Your Turn:** Try replacing `my_sentence` above with a sentence of your own or maybe something a student actually wrote, and re-run all the cells in this part.



---

### Part III: Counting and Weighting Words


Now that we have successfully clean our, let's count how often each word appears in the sentence. This is what we called **bag of words**, treating text as just a collection of words and counts, ignoring grammar and order.


In [ ]:
from collections import Counter # used to count all words in sentence

word_counts = Counter(content_words)
word_counts


One of the flaws of raw count data is that common words can dominate the counts across every piece of text. This flaw makes it hard to tell texts apart, but there's a solution to this flaw. **TF-IDF** (Term Frequency–Inverse Document Frequency) is our solution, which down-sizing words that show up in many documents, and upsizing words that are distinctive to just a few. It's one of the most widely used tools in real search engines and text classifiers.

Let's see it in action on three short, related sentences.


**Step 1: Set up a tiny, three-sentence "corpus"**

Corpus is a fancy word in NLP meaning a collection of documents. This corpus we have here will be used for us to see what TF-IDF does to each word.

In [ ]:
mini_corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat played with the dog"
]

mini_corpus


**Step 2: Fit a `TfidfVectorizer`**

Scikit-learn's [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) scans our text to build a vocabulary of every unique word, then calculates a TF-IDF score for each one based on how common or rare it is.

**Question to think about:** If our dataset contains a typo like "teh" instead of "the", how will TfidfVectorizer score it? Will it assign "teh" a high or low TF-IDF value, and why might that be a problem for our model?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(mini_corpus)

tfidf_matrix.shape # (number of sentences, number of unique words)

**Step 3: Turn the result into a readable table.**

By default, TfidfVectorizer outputs a memory-efficient "sparse matrix" that isn't easily human-readable. Let's convert this matrix into a standard pandas DataFrame using the unique words as column headers so we can easily inspect the scores.

In [ ]:
tfidf_table = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["doc 1", "doc 2", "doc 3"]
)
tfidf_table.style.format("{:.2f}")


<div class="alert alert-success">

**Considering the table above…**

* The word "the" appears in every sentence. What is its TF-IDF score, and why?

* Words like "mat," "log," and "played" each appear only once. How do their scores compare to "the"?

* Why might these weighted scores be more useful than raw counts for figuring out what makes a document distinctive?

</div>


**Summary:** You now know how to preprocess text by hand and with regular expressions, and how to turn text into numbers using bag of words and TF-IDF.

Let's apply these skills to a real, current question!


<hr style="border: 5px solid #003262;" />

# Can You Spot the Robot in the Room: An Investigation of AI-Generated Writing


Today, educators face a growing dilemma: how can you distinguish human writing from AI-generated text, and are detection tools actually trustworthy?

To find out, we'll build on the dataset we loaded in Part I, where we sampled 3,000 human and AI-generated articles from the larger dataset of over 17,000.

<hr style="border: 1px solid #fdb515;" />

## Part I: Human or AI?

🤔 Before we let the computer help, let's try it ourselves! Every row in `texts` already has a `label` telling us whether it's `Human`- or `AI`-written — the computer isn't figuring this out on the fly, we're just hiding that column from you for a moment so you can guess.

Run the cell below to see one random text with its label hidden. Try to guess `Human` or `AI` before running the next cell to reveal the answer — then come back and run both cells again for a new one.


In [ ]:
# grab a random text sammple
quiz_sample = texts.sample(1).iloc[0]

# grab sample text and it's correct answer
quiz_text = quiz_sample["text"]
correct_answer = quiz_sample["label"]

print("YOUR GENERATED TEXT IS --\n" + quiz_text)

In [ ]:
# Now for the correct answer
print("Your generated text was " + correct_answer)

Now that you have learned that the text was either AI or Human generated, run the cell below ONCE and the two cells above again. Do **NOT** run the cell below this text as you will overwrite your first generated text.

In [ ]:
first_text = quiz_text
first_text_answer = correct_answer

print("Your first generated text was: " + first_text + "\n\n The answer to your first generated text was: " + first_text_answer)

Now comparing the two texts what do you think your second generated text is?

**Some questions to think about:**

* What do the two texts have in common?
* What tone does each of the texts give off?
* Is there a rhythm to the sentences, and how long are they?



<hr style="border: 1px solid #fdb515;" />

### Part II: Visualizing Word Choice

Now let's apply what we learned in Part II and III: cleaning, tokenizing, and counting — across all 3,000 texts, and compare the most common words in `Human` versus `AI` writing.


**Step 1: Reuse our cleaning function, at scale.**

`clean_and_tokenize` runs the same regex-tokenize-and-remove-stopwords steps from Part II, but now we'll call it on every text in the whole dataset instead of just one sentence. `most_common_words` then tallies up the results for one label at a time.


In [ ]:
def clean_and_tokenize(text):
    tokens = regex_tokenize(text)
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS]

def most_common_words(label, n=10):
    all_words = []
    for text in texts.loc[texts["label"] == label, "text"]:
        all_words.extend(clean_and_tokenize(text))
    return Counter(all_words).most_common(n)

**Step 2: Run `Human` and `AI` texts separately.**

This is the part will take a bit longerto run, because it's cleaning and counting words across 3,000 articles.


In [ ]:
human_top_words = most_common_words("Human")
ai_top_words = most_common_words("AI")

print("Most common content words in HUMAN texts:", human_top_words)
print("Most common content words in AI texts:", ai_top_words)

**Step 3: Plot both lists side by side**

Observe the patterns of the two types of text.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (label, top_words) in zip(axes, [("Human", human_top_words), ("AI", ai_top_words)]):
    words, counts = zip(*top_words)
    ax.barh(words, counts, color="#003262" if label == "Human" else "#fdb515")
    ax.set_title(f"Most Common Content Words: {label}")
    ax.invert_yaxis()
plt.tight_layout()
plt.show()


**Some questions to think about:**
<div class="alert alert-success">

* What do you notice and wonder?

* What kinds of words show up in one list but not the other?

* Which of these patterns feel like real "signal," and which might just be quirks of this particular dataset?

</div>


<hr style="border: 1px solid #fdb515;" />

### Part III: Building a Detector

Word choice is one strategy. Let's also try a simple number: the average number of characters per word, as a rough stand-in for vocabulary sophistication. Does that alone separate the two groups?

**Before we create the plot, consider:** what do you expect the distributions to look like?


In [ ]:
# Average word length: total characters in the text, divided by number of words.
texts["avg_word_length"] = texts["text"].apply(lambda t: len(t.replace(" ", "")) / len(t.split()))

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(texts.loc[texts["label"] == "Human", "avg_word_length"], bins=20, alpha=0.6, label="Human", color="#003262")
plt.hist(texts.loc[texts["label"] == "AI", "avg_word_length"], bins=20, alpha=0.6, label="AI", color="#fdb515")
plt.title("Average Word Length by Label")
plt.xlabel("Average Word Length (characters)")
plt.ylabel("Number of Texts")
plt.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

* Is this what you expected? Why or why not?

* Do `Human` and `AI` look clearly separated, or is there overlap?

* Based on this chart, could you write a simple rule to guess `Human` or `AI`? What threshold would you pick?

</div>


Let's turn your guess into code: a **rule-based classifier**. If `avg_word_length` is above some threshold, guess `AI`; otherwise, guess `Human`.

The starting threshold below with 4.7 as that is roughly the average character of a word. You can adjust the value once you see the accuracy.

In [ ]:
threshold = 4.7

rule_preds = texts["avg_word_length"].apply(lambda score: "AI" if score > threshold else "Human")
rule_accuracy = (rule_preds == texts["label"]).mean()

print(f"Rule-based accuracy (threshold = {threshold}): {rule_accuracy:.1%}")

Instead of guessing a threshold by hand, what if the computer could learn its own rules from the data? We'll train a machine learning classifier. To make the test fair, we'll check it against texts it has never seen before, the same way you'd want to see how a student handles a brand-new piece of writing, not one they've already turned in.

**Step 1: Split into training and test sets.**

We hold back 20% of the texts as a test set the model never trains on. `stratify=texts["label"]` makes sure both sets keep roughly the same Human/AI balance as the full dataset.


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(texts, test_size=0.2, random_state=42, stratify=texts["label"])

print("Training examples:", len(train_df))
print("Test examples:    ", len(test_df))


**Step 2: Turn every text into TF-IDF numbers.**

This is the same tool from Part III, just applied to real articles instead of toy sentences. Notice we `fit_transform` on the training text only, then `transform` the test text in which the vectorizer should only ever learn its vocabulary from training data, the same way the model itself does.


In [ ]:
ml_vectorizer = TfidfVectorizer(stop_words="english")
X_train = ml_vectorizer.fit_transform(train_df["text"])
X_test = ml_vectorizer.transform(test_df["text"])

X_train.shape # (number of training texts, number of unique words)


**Step 3: Train a `LogisticRegression` classifier.**

This is a standard, well-understood machine learning model, in which it learns a weight for every word in the vocabulary, indicating how strongly that word points toward `Human` or `AI`.


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_df["label"])


**Step 4: Use the trained model to predict labels for the test set**

And observe score how often it got it right.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

ml_preds = model.predict(X_test)
ml_accuracy = accuracy_score(test_df["label"], ml_preds)

print(f"ML classifier accuracy on test set: {ml_accuracy:.1%}")


**Step 5: Compare to the simple rule from Part III**

It evaluated on this same held-out test set, so it's a fair, apples-to-apples comparison.


In [ ]:
rule_preds_test = test_df["avg_word_length"].apply(lambda score: "AI" if score > threshold else "Human")
rule_accuracy_test = accuracy_score(test_df["label"], rule_preds_test)

print(f"Rule-based accuracy on test set: {rule_accuracy_test:.1%}")
print(f"ML classifier accuracy on test set:  {ml_accuracy:.1%}")


Note: We typically won't see such a high score in real-world use, because most AI-written text doesn't carry obvious giveaways like this dataset does.

But before we end, let's take a closer look at exactly where the model was right and wrong, using what we called a confusion matrix.


In [ ]:
# A confusion matrix shows exactly where the model was right and wrong.
cm = confusion_matrix(test_df["label"], ml_preds, labels=["Human", "AI"])
pd.DataFrame(cm, index=["Actually Human", "Actually AI"], columns=["Predicted Human", "Predicted AI"])


<hr style="border: 1px solid #fdb515;" />


## In the Classroom

This notebook models a core scientific activity: cleaning, structuring, and analyzing text data to find patterns, then questioning how reliable and fair those patterns really are.

**Where else do we see this kind of analysis?**

The skills you just practiced of tokenizing text, weighting words with TF-IDF, and training and auditing a classifier are the foundational steps to analyzing any collection of written language.

Think about how you could apply this same skill sets in other areas:

* **English Language Arts:** Compare word choice and tone across authors, speeches, or historical periods.

* **Social Studies:** Analyze framing and bias in news headlines from different sources.

* **Any Subject:** Have students analyze the vocabulary and tone of their own writing over the course of a semester.

💭 Any time you ask a question involving **"is this real, or generated by an LLM,"** these tools and concepts are a strong place to start.

---


## Credits

This notebook was developed as part of "Show Your Work" (SyW), a research and development project at UC Berkeley to introduce computational notebooks to K-12 educators.

[Acknowledge contributors, data sources, image sources, and funding here.]
